<div style="text-align: center;">
    <h1>Random Forest — Clasificación de Enfermedad Cardiovascular</h1>
</div>

## 1. Importar librerías

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn shap -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    train_test_split, RandomizedSearchCV, StratifiedKFold,
    learning_curve, validation_curve
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve
)

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42

## 2. Cargar datos

Se carga directamente la base de datos ya limpia y preprocesada desde el repositorio GitHub del proyecto.

In [ ]:
url = "https://raw.githubusercontent.com/SantCorrea802/Cardiovascular_Disease_Proyecto_Modelos_II/main/dataset/data_cleaned.csv"
data_clean = pd.read_csv(url)

print(f"Shape: {data_clean.shape}")
data_clean.head()

## 3. Separación Train / Validation / Test

Se usa la misma estrategia del EDA: **80% train — 10% val — 10% test**, con estratificación para mantener la proporción de clases.

In [ ]:
features = data_clean.drop(columns=["cardiovascular_disease"])
target   = data_clean["cardiovascular_disease"]

X_train, X_temp, y_train, y_temp = train_test_split(
    features, target, test_size=0.20, random_state=RANDOM_STATE, stratify=target
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print(f"Train:      {X_train.shape[0]} muestras")
print(f"Validation: {X_val.shape[0]} muestras")
print(f"Test:       {X_test.shape[0]} muestras")

## 4. Modelo base (baseline)

Se entrena un Random Forest con hiperparámetros por defecto para establecer una línea base de comparación.

In [ ]:
rf_baseline = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
rf_baseline.fit(X_train, y_train)

y_val_pred_base = rf_baseline.predict(X_val)
y_val_prob_base = rf_baseline.predict_proba(X_val)[:, 1]

print("=== Baseline — Validación ===")
print(f"Accuracy:  {accuracy_score(y_val, y_val_pred_base):.4f}")
print(f"Precision: {precision_score(y_val, y_val_pred_base):.4f}")
print(f"Recall:    {recall_score(y_val, y_val_pred_base):.4f}")
print(f"F1-Score:  {f1_score(y_val, y_val_pred_base):.4f}")
print(f"AUC-ROC:   {roc_auc_score(y_val, y_val_prob_base):.4f}")

## 5. Búsqueda de hiperparámetros — RandomizedSearchCV

Se usa `RandomizedSearchCV` con validación cruzada estratificada (5 folds) sobre el conjunto de entrenamiento. Para cada combinación de hiperparámetros se calcula el AUC-ROC promedio en los 5 folds, y se selecciona la combinación con mejor resultado.

### Malla de hiperparámetros

| Hiperparámetro | Descripción | Valores explorados |
|---|---|---|
| `n_estimators` | Número de árboles en el bosque | [100, 200, 300, 500] |
| `max_depth` | Profundidad máxima de cada árbol | [None, 10, 20, 30] |
| `min_samples_split` | Mínimo de muestras para dividir un nodo | [2, 5, 10] |
| `min_samples_leaf` | Mínimo de muestras en una hoja | [1, 2, 4] |
| `max_features` | Número de features a considerar por split | ['sqrt', 'log2'] |
| `class_weight` | Peso de las clases | [None, 'balanced'] |

In [ ]:
param_dist = {
    "n_estimators":      [100, 200, 300, 500],
    "max_depth":         [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf":  [1, 2, 4],
    "max_features":      ["sqrt", "log2"],
    "class_weight":      [None, "balanced"]
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=40,
    scoring="roc_auc",
    cv=cv_strategy,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_train, y_train)

print("\nMejores hiperparámetros encontrados:")
for param, value in rf_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nMejor AUC-ROC (CV): {rf_search.best_score_:.4f}")

## 6. Evaluación del modelo optimizado en Validación

In [ ]:
rf_best = rf_search.best_estimator_

y_val_pred = rf_best.predict(X_val)
y_val_prob = rf_best.predict_proba(X_val)[:, 1]

metrics_val = {
    "Accuracy":  accuracy_score(y_val, y_val_pred),
    "Precision": precision_score(y_val, y_val_pred),
    "Recall":    recall_score(y_val, y_val_pred),
    "F1-Score":  f1_score(y_val, y_val_pred),
    "AUC-ROC":   roc_auc_score(y_val, y_val_prob)
}

print("=== Modelo Optimizado — Validación ===")
for metric, value in metrics_val.items():
    print(f"{metric:10s}: {value:.4f}")

### Matriz de confusión y curva ROC — Validación

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_val, y_val_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Sin cardiopatía", "Con cardiopatía"])
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Matriz de Confusión — Validación")

fpr, tpr, _ = roc_curve(y_val, y_val_prob)
auc = roc_auc_score(y_val, y_val_prob)
axes[1].plot(fpr, tpr, color="steelblue", lw=2, label=f"AUC = {auc:.4f}")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("Tasa de Falsos Positivos")
axes[1].set_ylabel("Tasa de Verdaderos Positivos")
axes[1].set_title("Curva ROC — Validación")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

## 7. Detección de sobre/subajuste

### 7.1 Curva de aprendizaje

Muestra cómo evolucionan las métricas de **train** y **validación cruzada** al aumentar el tamaño del conjunto de entrenamiento.

- **Train alto, val bajo** → overfitting (sobreajuste)
- **Ambos bajos** → underfitting (subajuste)
- **Ambos altos y cercanos** → modelo bien ajustado ✅

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    rf_best,
    X_train, y_train,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring="roc_auc",
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

plt.figure(figsize=(9, 5))
plt.plot(train_sizes, train_mean, "o-", color="steelblue", label="Train")
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color="steelblue")
plt.plot(train_sizes, val_mean, "o-", color="tomato", label="Validación cruzada")
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color="tomato")
plt.xlabel("Tamaño del conjunto de entrenamiento")
plt.ylabel("AUC-ROC")
plt.title("Curva de Aprendizaje — Random Forest")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

gap = train_mean[-1] - val_mean[-1]
print(f"AUC-ROC Train (completo):      {train_mean[-1]:.4f} ± {train_std[-1]:.4f}")
print(f"AUC-ROC Validación (completo): {val_mean[-1]:.4f} ± {val_std[-1]:.4f}")
print(f"Brecha train-val:              {gap:.4f}")
if gap > 0.05:
    print("⚠️  Posible sobreajuste (brecha > 0.05)")
elif val_mean[-1] < 0.70:
    print("⚠️  Posible subajuste (AUC-ROC val < 0.70)")
else:
    print("✅ Modelo bien ajustado")

### 7.2 Curva de validación — efecto de `max_depth`

Muestra cómo varía el desempeño al cambiar la profundidad máxima del árbol.

- Si train sube pero val baja al aumentar la profundidad → **overfitting**
- Si ambos son bajos con poca profundidad → **underfitting**

In [ ]:
best_params = rf_search.best_params_.copy()
best_params.pop("max_depth")

depth_range = [3, 5, 10, 15, 20, 30, None]
depth_labels = [str(d) if d is not None else "None" for d in depth_range]

train_scores_vc, val_scores_vc = validation_curve(
    RandomForestClassifier(**best_params, random_state=RANDOM_STATE, n_jobs=-1),
    X_train, y_train,
    param_name="max_depth",
    param_range=depth_range,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring="roc_auc",
    n_jobs=-1
)

train_m = train_scores_vc.mean(axis=1)
train_s = train_scores_vc.std(axis=1)
val_m   = val_scores_vc.mean(axis=1)
val_s   = val_scores_vc.std(axis=1)

x = np.arange(len(depth_range))
plt.figure(figsize=(9, 5))
plt.plot(x, train_m, "o-", color="steelblue", label="Train")
plt.fill_between(x, train_m - train_s, train_m + train_s, alpha=0.15, color="steelblue")
plt.plot(x, val_m, "o-", color="tomato", label="Validación cruzada")
plt.fill_between(x, val_m - val_s, val_m + val_s, alpha=0.15, color="tomato")
plt.xticks(x, depth_labels)
plt.xlabel("max_depth")
plt.ylabel("AUC-ROC")
plt.title("Curva de Validación — Efecto de max_depth")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7.3 Curva de validación — convergencia del ensemble (`n_estimators`)

Muestra en qué punto agregar más árboles deja de mejorar el modelo. Cuando la curva de validación se aplana, ese es el tope óptimo.

In [ ]:
n_range = [10, 50, 100, 200, 300, 500]

train_scores_n, val_scores_n = validation_curve(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    X_train, y_train,
    param_name="n_estimators",
    param_range=n_range,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring="roc_auc",
    n_jobs=-1
)

train_mn = train_scores_n.mean(axis=1)
train_sn = train_scores_n.std(axis=1)
val_mn   = val_scores_n.mean(axis=1)
val_sn   = val_scores_n.std(axis=1)

plt.figure(figsize=(9, 5))
plt.plot(n_range, train_mn, "o-", color="steelblue", label="Train")
plt.fill_between(n_range, train_mn - train_sn, train_mn + train_sn, alpha=0.15, color="steelblue")
plt.plot(n_range, val_mn, "o-", color="tomato", label="Validación cruzada")
plt.fill_between(n_range, val_mn - val_sn, val_mn + val_sn, alpha=0.15, color="tomato")
plt.xlabel("n_estimators (número de árboles)")
plt.ylabel("AUC-ROC")
plt.title("Curva de Validación — Convergencia del ensemble (n_estimators)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

mejoras = np.diff(val_mn)
convergencia = n_range[np.argmax(mejoras < 0.001) + 1]
print(f"El modelo converge aproximadamente en n_estimators = {convergencia}")
print("A partir de ahí, agregar más árboles no mejora significativamente el AUC-ROC.")

## 8. Importancia de features

In [ ]:
importances = pd.Series(rf_best.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(9, 6))
importances.plot(kind="barh", color="steelblue", edgecolor="white")
plt.title("Importancia de Features — Random Forest")
plt.xlabel("Importancia (Gini)")
plt.tight_layout()
plt.show()

print(importances.sort_values(ascending=False).round(4))

## 9. Explicabilidad con SHAP

SHAP (SHapley Additive exPlanations) asigna a cada feature un valor que indica **cuánto y en qué dirección** contribuyó a la predicción del modelo. Se calcula sobre una muestra aleatoria del train para mayor velocidad.

In [ ]:
X_shap = X_train.sample(n=500, random_state=RANDOM_STATE)

explainer   = shap.TreeExplainer(rf_best)
shap_values = explainer.shap_values(X_shap)
shap_vals_class1 = shap_values[1]

print(f"SHAP values calculados sobre {X_shap.shape[0]} muestras.")

### 9.1 Summary Plot (Beeswarm)

Cada punto es una observación. El color indica el valor de la feature (rojo = alto, azul = bajo). La posición en X indica si la feature **aumenta o disminuye** la probabilidad de cardiopatía.

In [ ]:
shap.summary_plot(shap_vals_class1, X_shap, plot_type="dot", show=True)

### 9.2 Bar Plot — Importancia global promedio

Muestra el impacto promedio absoluto de cada feature sobre todas las predicciones.

In [ ]:
shap.summary_plot(shap_vals_class1, X_shap, plot_type="bar", show=True)

### 9.3 Waterfall Plot — Explicación de una predicción individual

Muestra cómo cada feature empuja la predicción hacia arriba o hacia abajo para **un paciente específico**.

In [ ]:
idx = 0
shap.plots._waterfall.waterfall_legacy(
    explainer.expected_value[1],
    shap_vals_class1[idx],
    feature_names=X_shap.columns.tolist(),
    max_display=10,
    show=True
)

pred_prob  = rf_best.predict_proba(X_shap.iloc[[idx]])[0, 1]
pred_class = rf_best.predict(X_shap.iloc[[idx]])[0]
print(f"\nPaciente #{idx}: probabilidad de cardiopatía = {pred_prob:.4f} → clase predicha = {pred_class}")

## 10. Resumen de hiperparámetros explorados

Tabla para incluir en el informe IEEE.

In [ ]:
resumen_hiperparametros = pd.DataFrame([
    {"Hiperparámetro": "n_estimators",      "Descripción": "Número de árboles",                     "Malla de valores": "[100, 200, 300, 500]"},
    {"Hiperparámetro": "max_depth",          "Descripción": "Profundidad máxima del árbol",           "Malla de valores": "[None, 10, 20, 30]"},
    {"Hiperparámetro": "min_samples_split",  "Descripción": "Mín. muestras para dividir un nodo",    "Malla de valores": "[2, 5, 10]"},
    {"Hiperparámetro": "min_samples_leaf",   "Descripción": "Mín. muestras en hoja terminal",        "Malla de valores": "[1, 2, 4]"},
    {"Hiperparámetro": "max_features",       "Descripción": "Features consideradas por split",       "Malla de valores": "['sqrt', 'log2']"},
    {"Hiperparámetro": "class_weight",       "Descripción": "Peso de las clases",                    "Malla de valores": "[None, 'balanced']"},
])

print("=== Tabla de Hiperparámetros — Random Forest ===")
print(resumen_hiperparametros.to_string(index=False))

print("\n=== Mejor combinación encontrada ===")
for k, v in rf_search.best_params_.items():
    print(f"  {k}: {v}")

## 11. Evaluación final en Test

Se evalúa el modelo optimizado sobre el conjunto de test. Esta celda refleja el desempeño real del modelo sobre datos que no participaron en ninguna etapa de entrenamiento ni selección de hiperparámetros.

In [ ]:
y_test_pred = rf_best.predict(X_test)
y_test_prob = rf_best.predict_proba(X_test)[:, 1]

# Métricas en train
y_train_pred = rf_best.predict(X_train)
y_train_prob = rf_best.predict_proba(X_train)[:, 1]

metrics_test = {
    "Accuracy":  accuracy_score(y_test, y_test_pred),
    "Precision": precision_score(y_test, y_test_pred),
    "Recall":    recall_score(y_test, y_test_pred),
    "F1-Score":  f1_score(y_test, y_test_pred),
    "AUC-ROC":   roc_auc_score(y_test, y_test_prob)
}

# Tabla comparativa Train / Validación / Test
comparison = pd.DataFrame({
    "Train": [
        accuracy_score(y_train, y_train_pred),
        precision_score(y_train, y_train_pred),
        recall_score(y_train, y_train_pred),
        f1_score(y_train, y_train_pred),
        roc_auc_score(y_train, y_train_prob)
    ],
    "Validación": list(metrics_val.values()),
    "Test": list(metrics_test.values())
}, index=["Accuracy", "Precision", "Recall", "F1-Score", "AUC-ROC"])

print("=== Comparativa Train / Validación / Test ===")
print(comparison.round(4))

### Matriz de confusión y curva ROC — Test

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_test = confusion_matrix(y_test, y_test_pred)
disp_test = ConfusionMatrixDisplay(confusion_matrix=cm_test, display_labels=["Sin cardiopatía", "Con cardiopatía"])
disp_test.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Matriz de Confusión — Test")

fpr_t, tpr_t, _ = roc_curve(y_test, y_test_prob)
auc_t = roc_auc_score(y_test, y_test_prob)
axes[1].plot(fpr_t, tpr_t, color="steelblue", lw=2, label=f"AUC = {auc_t:.4f}")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("Tasa de Falsos Positivos")
axes[1].set_ylabel("Tasa de Verdaderos Positivos")
axes[1].set_title("Curva ROC — Test")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()